# Linguagens de Programação para Engenharia de Dados
## Encontro 3 — Notebook Prático: SQL para Engenharia de Dados

**Instituição:** Universidade de Fortaleza (UNIFOR)
**Curso:** Pós-Graduação — Especialização em Engenharia de Dados
**Professor:** Cassio Pinheiro — [LinkedIn](https://www.linkedin.com/in/cassio-pinheiro-9baa7939/)
**Data:** 27/06/2026 · 19:00 às 22:30

---

### Como usar este notebook

1. Execute as células **na ordem**, de cima para baixo (`Shift + Enter`).
2. Cada seção corresponde a uma seção do documento `.md` do encontro.
3. Todo o SQL roda no **DuckDB**, um banco analítico que vive *dentro do notebook* (in-process). Nada de servidor para instalar.
4. **Mexa nas consultas.** Troque colunas, filtros e ordens; quebre de propósito; conserte. É assim que se aprende SQL.

> **Projeto fio condutor:** o `vendas.csv` dos encontros anteriores vira um pequeno **modelo estrela** — uma tabela-fato `vendas` cercada pelas dimensões `clientes` e `categorias`. Toda consulta de hoje acontece sobre esse modelo.

## Setup — DuckDB + pandas

`duckdb` é leve, in-process e instalável com um `pip`. `pandas` já é nosso conhecido do Encontro 2. `pyarrow` entra só na seção 6, para escrever/ler Parquet.

In [1]:
import duckdb
import pandas as pd

print("DuckDB versao :", duckdb.__version__)
print("pandas versao :", pd.__version__)
print("Setup OK — SQL roda dentro do proprio notebook, sem servidor.")

DuckDB versao : 1.5.4
pandas versao : 2.3.3
Setup OK — SQL roda dentro do proprio notebook, sem servidor.


---

# 1 — Pensar em conjuntos

> 📖 **No `.md`:** seção *1 — Pensar em conjuntos*.

Primeiro montamos o **modelo estrela** sintético e determinístico. Partimos do mesmo `vendas.csv` *sujo* do Encontro 1, limpamos com pandas (o que já sabemos fazer) e carregamos no DuckDB como a tabela-fato `vendas`. Depois criamos as dimensões `clientes` e `categorias`.

In [2]:
import io

# Mesmo CSV bruto do Encontro 1 — sujo de proposito.
csv_bruto = """id,data,cliente_id,categoria_id,valor
1,2026-06-01,1,10,1200.00
2,01/06/2026,2,20,85.50
3,2026-06-02,3,10,
4,2026-06-02,4,30,-30.00
5,2026/06/03,1,20,42.90
5,2026/06/03,1,20,42.90
6,2026-06-03,5,40,260.00
7,,6,50,99.90
8,2026-06-04,2,10,540.00
9,2026-06-04,1,10,310.00
10,2026-06-05,5,20,73.20
11,2026-06-05,3,40,180.00
12,2026-06-06,2,30,45.00
"""

bruto = pd.read_csv(io.StringIO(csv_bruto), dtype=str)
print("Linhas brutas:", len(bruto))
bruto

Linhas brutas: 13


,id,data,cliente_id,categoria_id,valor
0,1,2026-06-01,1,10,1200.00
1,2,01/06/2026,2,20,85.50
2,3,2026-06-02,3,10,NaN
3,4,2026-06-02,4,30,-30.00
4,5,2026/06/03,1,20,42.90
5,5,2026/06/03,1,20,42.90
6,6,2026-06-03,5,40,260.00
7,7,NaN,6,50,99.90
8,8,2026-06-04,2,10,540.00
9,9,2026-06-04,1,10,310.00


In [3]:
# --- Limpeza com pandas (revisao do Encontro 2) ---
df = bruto.copy()

# data: aceita 3 formatos -> normaliza para datetime; descarta data invalida/vazia
df["data"] = pd.to_datetime(df["data"], format="mixed", dayfirst=True, errors="coerce")

# valor: texto -> numerico; vazio vira NaN
df["valor"] = pd.to_numeric(df["valor"], errors="coerce")

# ids para inteiro
df["id"] = df["id"].astype(int)
df["cliente_id"] = df["cliente_id"].astype(int)
df["categoria_id"] = df["categoria_id"].astype(int)

antes = len(df)
df = df.dropna(subset=["data", "valor"])          # remove valor/data ausentes
df = df[df["valor"] >= 0]                          # remove negativos
df = df.drop_duplicates(subset=["id"])            # remove duplicata por id
df = df.reset_index(drop=True)

print(f"Limpeza: {antes} -> {len(df)} linhas mantidas (removidos ausentes, negativos e duplicata).")
vendas = df
vendas

Limpeza: 13 -> 9 linhas mantidas (removidos ausentes, negativos e duplicata).


,id,data,cliente_id,categoria_id,valor
0,1,2026-06-01,1,10,1200.0
1,2,2026-06-01,2,20,85.5
2,5,2026-06-03,1,20,42.9
3,6,2026-06-03,5,40,260.0
4,8,2026-06-04,2,10,540.0
5,9,2026-06-04,1,10,310.0
6,10,2026-06-05,5,20,73.2
7,11,2026-06-05,3,40,180.0
8,12,2026-06-06,2,30,45.0


In [4]:
# --- Dimensoes do modelo estrela ---
clientes = pd.DataFrame({
    "id":   [1, 2, 3, 4, 5, 6, 7],
    "nome": ["Ana Souza", "Bruno Lima", "Carla Dias", "Diego Reis",
             "Elena Cruz", "Felipe Aragao", "Gabriela Nunes"],
    "cidade":   ["Fortaleza", "Recife", "Fortaleza", "Natal",
                 "Fortaleza", "Recife", "Sobral"],
    "segmento": ["varejo", "varejo", "corporativo", "varejo",
                 "corporativo", "varejo", "corporativo"],
})

categorias = pd.DataFrame({
    "id":   [10, 20, 30, 40, 50],
    "nome": ["eletronicos", "alimentacao", "transporte", "vestuario", "servicos"],
    "departamento": ["tecnologia", "consumo", "logistica", "consumo", "operacoes"],
})

print("Dimensao clientes:")
print(clientes)
print("\nDimensao categorias:")
print(categorias)
print("\nNote: o cliente 7 (Gabriela) NAO tem vendas — sera util nos JOINs.")

Dimensao clientes:
   id            nome     cidade     segmento
0   1       Ana Souza  Fortaleza       varejo
1   2      Bruno Lima     Recife       varejo
2   3      Carla Dias  Fortaleza  corporativo
3   4      Diego Reis      Natal       varejo
4   5      Elena Cruz  Fortaleza  corporativo
5   6   Felipe Aragao     Recife       varejo
6   7  Gabriela Nunes     Sobral  corporativo

Dimensao categorias:
   id         nome departamento
0  10  eletronicos   tecnologia
1  20  alimentacao      consumo
2  30   transporte    logistica
3  40    vestuario      consumo
4  50     servicos    operacoes

Note: o cliente 7 (Gabriela) NAO tem vendas — sera util nos JOINs.


In [5]:
# --- Carrega o modelo estrela no DuckDB ---
con = duckdb.connect()                 # banco in-process (na memoria)
con.register("vendas", vendas)         # a fato
con.register("clientes", clientes)     # dimensao
con.register("categorias", categorias) # dimensao

print("Tabelas registradas no DuckDB:")
print(con.sql("SHOW TABLES").df())

Tabelas registradas no DuckDB:
         name
0  categorias
1    clientes
2      vendas


**A mesma pergunta, duas mentalidades.** *"Qual o faturamento por categoria?"*

Primeiro **procedural** (Python, passo a passo), depois **declarativo** (SQL, descrevendo o resultado):

In [6]:
# (a) PROCEDURAL — descrevemos o caminho: percorrer e acumular
faturamento = {}
for _, linha in vendas.iterrows():
    cat = linha["categoria_id"]
    faturamento[cat] = faturamento.get(cat, 0.0) + linha["valor"]
print("Procedural (Python):", {k: round(v, 2) for k, v in faturamento.items()})

# (b) DECLARATIVO — descrevemos o destino: o banco escolhe o caminho
print("\nDeclarativo (SQL via DuckDB):")
con.sql("""
    SELECT categoria_id, SUM(valor) AS faturamento
    FROM vendas
    GROUP BY categoria_id
    ORDER BY faturamento DESC
""").df()

Procedural (Python): {10: 2050.0, 20: 201.6, 40: 440.0, 30: 45.0}

Declarativo (SQL via DuckDB):


,categoria_id,faturamento
0,10,2050.0
1,40,440.0
2,20,201.6
3,30,45.0


---

# 2 — SELECT essencial

> 📖 **No `.md`:** seção *2 — SELECT essencial*.

`SELECT / WHERE / ORDER BY / LIMIT`, tipos com `CAST` e valores únicos com `DISTINCT`.
Toda consulta abaixo é executada com `con.sql(...).df()`, devolvendo um DataFrame.

In [7]:
# SELECT + WHERE + ORDER BY + LIMIT
con.sql("""
    SELECT id, cliente_id, categoria_id, valor, data
    FROM vendas
    WHERE valor > 100          -- filtra LINHAS antes de tudo
    ORDER BY valor DESC        -- da maior para a menor
    LIMIT 5                    -- so as 5 primeiras
""").df()

,id,cliente_id,categoria_id,valor,data
0,1,1,10,1200.0,2026-06-01
1,8,2,10,540.0,2026-06-04
2,9,1,10,310.0,2026-06-04
3,6,5,40,260.0,2026-06-03
4,11,3,40,180.0,2026-06-05


In [8]:
# CAST — converter tipos explicitamente, e DISTINCT — valores unicos
print("CAST: valor para INTEGER (trunca casas) e data para texto:")
print(con.sql("""
    SELECT id,
           CAST(valor AS INTEGER) AS valor_inteiro,
           CAST(data AS DATE)     AS data_real
    FROM vendas
    ORDER BY id
    LIMIT 4
""").df())

print("\nDISTINCT: categorias presentes na fato (o que um set faria em Python):")
print(con.sql("SELECT DISTINCT categoria_id FROM vendas ORDER BY categoria_id").df())

CAST: valor para INTEGER (trunca casas) e data para texto:
   id  valor_inteiro  data_real
0   1           1200 2026-06-01
1   2             86 2026-06-01
2   5             43 2026-06-03
3   6            260 2026-06-03

DISTINCT: categorias presentes na fato (o que um set faria em Python):
   categoria_id
0            10
1            20
2            30
3            40


---

# 3 — Agregação

> 📖 **No `.md`:** seção *3 — Agregação*.

`GROUP BY` + `SUM/AVG/COUNT/MIN/MAX`, filtro de grupos com `HAVING` e classificação com `CASE WHEN`.

In [9]:
# GROUP BY com varias funcoes de agregacao
con.sql("""
    SELECT categoria_id,
           COUNT(*)              AS qtd_vendas,
           SUM(valor)            AS faturamento,
           ROUND(AVG(valor), 2)  AS ticket_medio,
           MIN(valor)            AS menor_venda,
           MAX(valor)            AS maior_venda
    FROM vendas
    GROUP BY categoria_id
    ORDER BY faturamento DESC
""").df()

,categoria_id,qtd_vendas,faturamento,ticket_medio,menor_venda,maior_venda
0,10,3,2050.0,683.33,310.0,1200.0
1,40,2,440.0,220.00,180.0,260.0
2,20,3,201.6,67.20,42.9,85.5
3,30,1,45.0,45.00,45.0,45.0


In [10]:
# HAVING — filtra GRUPOS depois da agregacao (WHERE nao enxerga SUM)
print("Categorias que faturaram mais de R$ 300:")
print(con.sql("""
    SELECT categoria_id, SUM(valor) AS faturamento
    FROM vendas
    GROUP BY categoria_id
    HAVING SUM(valor) > 300
    ORDER BY faturamento DESC
""").df())

Categorias que faturaram mais de R$ 300:
   categoria_id  faturamento
0            10       2050.0
1            40        440.0


In [11]:
# CASE WHEN — o if/elif/else do SQL: classifica cada venda em faixas
con.sql("""
    SELECT id, valor,
           CASE
             WHEN valor >= 1000 THEN 'alto valor'
             WHEN valor >= 100  THEN 'valor medio'
             ELSE 'baixo valor'
           END AS faixa
    FROM vendas
    ORDER BY valor DESC
""").df()

,id,valor,faixa
0,1,1200.0,alto valor
1,8,540.0,valor medio
2,9,310.0,valor medio
3,6,260.0,valor medio
4,11,180.0,valor medio
5,2,85.5,baixo valor
6,10,73.2,baixo valor
7,12,45.0,baixo valor
8,5,42.9,baixo valor


---

# 4 — JOINs

> 📖 **No `.md`:** seção *4 — JOINs*.

Costurar a fato `vendas` às dimensões. O que muda entre os tipos é **o que acontece com as linhas sem par**. Lembre-se: a cliente **Gabriela (id 7) não tem vendas** e a categoria também é completa — vamos provocar os casos.

In [12]:
# INNER JOIN — so o que casa nos DOIS lados (vendas + nome do cliente + categoria)
con.sql("""
    SELECT v.id, c.nome AS cliente, cat.nome AS categoria, v.valor
    FROM vendas v
    INNER JOIN clientes c    ON v.cliente_id   = c.id
    INNER JOIN categorias cat ON v.categoria_id = cat.id
    ORDER BY v.valor DESC
    LIMIT 6
""").df()

,id,cliente,categoria,valor
0,1,Ana Souza,eletronicos,1200.0
1,8,Bruno Lima,eletronicos,540.0
2,9,Ana Souza,eletronicos,310.0
3,6,Elena Cruz,vestuario,260.0
4,11,Carla Dias,vestuario,180.0
5,2,Bruno Lima,alimentacao,85.5


In [13]:
# LEFT JOIN — TODOS os clientes, mesmo os que nunca compraram.
# Gabriela (id 7) aparece com faturamento NULL -> 0.
con.sql("""
    SELECT c.nome AS cliente,
           COUNT(v.id)               AS qtd_vendas,
           COALESCE(SUM(v.valor), 0) AS faturamento
    FROM clientes c
    LEFT JOIN vendas v ON v.cliente_id = c.id
    GROUP BY c.nome
    ORDER BY faturamento DESC
""").df()

,cliente,qtd_vendas,faturamento
0,Ana Souza,3,1552.9
1,Bruno Lima,3,670.5
2,Elena Cruz,2,333.2
3,Carla Dias,1,180.0
4,Gabriela Nunes,0,0.0
5,Diego Reis,0,0.0
6,Felipe Aragao,0,0.0


In [14]:
# RIGHT JOIN — espelho do LEFT: preserva TODAS as linhas da tabela da direita (clientes).
# Mesmo resultado conceitual, invertendo os lados.
print("RIGHT JOIN (clientes a direita preservados):")
print(con.sql("""
    SELECT c.nome AS cliente, COUNT(v.id) AS qtd_vendas
    FROM vendas v
    RIGHT JOIN clientes c ON v.cliente_id = c.id
    GROUP BY c.nome
    ORDER BY qtd_vendas DESC
""").df())

RIGHT JOIN (clientes a direita preservados):
          cliente  qtd_vendas
0      Bruno Lima           3
1       Ana Souza           3
2      Elena Cruz           2
3      Carla Dias           1
4      Diego Reis           0
5   Felipe Aragao           0
6  Gabriela Nunes           0


In [15]:
# FULL OUTER JOIN — preserva os dois lados.
# Detecta inconsistencias: cliente sem venda (Gabriela) E venda orfa, se houvesse.
con.sql("""
    SELECT c.nome AS cliente, v.id AS venda_id, v.valor
    FROM clientes c
    FULL OUTER JOIN vendas v ON v.cliente_id = c.id
    ORDER BY cliente, venda_id
""").df()

,cliente,venda_id,valor
0,Ana Souza,1,1200.0
1,Ana Souza,5,42.9
2,Ana Souza,9,310.0
3,Bruno Lima,2,85.5
4,Bruno Lima,8,540.0
5,Bruno Lima,12,45.0
6,Carla Dias,11,180.0
7,Diego Reis,<NA>,NaN
8,Elena Cruz,6,260.0
9,Elena Cruz,10,73.2


---

# 5 — Window functions e CTEs

> 📖 **No `.md`:** seção *5 — Window functions e CTEs*.

Window functions trazem o resumo **sem colapsar** as linhas (`ROW_NUMBER`, `RANK`, `SUM() OVER`). CTEs (`WITH`) dão nome às etapas. O exemplo canônico: **ranking de clientes por faturamento**.

In [16]:
# SUM() OVER (PARTITION BY ...) — total da categoria EM CADA linha, sem perder o detalhe.
# E o % que cada venda representa dentro da sua categoria.
con.sql("""
    SELECT id, categoria_id, valor,
           SUM(valor) OVER (PARTITION BY categoria_id)                       AS total_categoria,
           ROUND(100 * valor / SUM(valor) OVER (PARTITION BY categoria_id), 1) AS pct_da_categoria
    FROM vendas
    ORDER BY categoria_id, valor DESC
""").df()

,id,categoria_id,valor,total_categoria,pct_da_categoria
0,1,10,1200.0,2050.0,58.5
1,8,10,540.0,2050.0,26.3
2,9,10,310.0,2050.0,15.1
3,2,20,85.5,201.6,42.4
4,10,20,73.2,201.6,36.3
5,5,20,42.9,201.6,21.3
6,12,30,45.0,45.0,100.0
7,6,40,260.0,440.0,59.1
8,11,40,180.0,440.0,40.9


In [17]:
# CTE (WITH) + RANK() — ranking de clientes por faturamento, legivel de cima para baixo.
con.sql("""
    WITH faturamento_cliente AS (
        SELECT c.nome,
               SUM(v.valor) AS faturamento,
               COUNT(*)     AS qtd_vendas
        FROM vendas v
        JOIN clientes c ON v.cliente_id = c.id
        GROUP BY c.nome
    )
    SELECT nome,
           faturamento,
           qtd_vendas,
           ROW_NUMBER() OVER (ORDER BY faturamento DESC) AS linha,
           RANK()       OVER (ORDER BY faturamento DESC) AS posicao
    FROM faturamento_cliente
    ORDER BY posicao
""").df()

,nome,faturamento,qtd_vendas,linha,posicao
0,Ana Souza,1552.9,3,1,1
1,Bruno Lima,670.5,3,2,2
2,Elena Cruz,333.2,2,3,3
3,Carla Dias,180.0,1,4,4


---

# 6 — SQL + Python na prática

> 📖 **No `.md`:** seção *6 — SQL + Python*.

O DuckDB consulta **DataFrames pandas** e **arquivos Parquet** diretamente. Aqui mostramos: consultar um DataFrame na memória, registrar explicitamente, e ler um Parquet do disco com `read_parquet` — sem ETL prévio.

In [18]:
# (a) DuckDB consultando um DataFrame pandas direto pelo nome da variavel
resumo = vendas.copy()
print("DuckDB le o DataFrame 'resumo' pelo nome, sem registrar:")
duckdb.sql("""
    SELECT categoria_id, COUNT(*) AS n, SUM(valor) AS faturamento
    FROM resumo
    GROUP BY categoria_id
    ORDER BY faturamento DESC
""").df()

DuckDB le o DataFrame 'resumo' pelo nome, sem registrar:


,categoria_id,n,faturamento
0,10,3,2050.0
1,40,2,440.0
2,20,3,201.6
3,30,1,45.0


In [19]:
# (b) Escrever a fato em Parquet e ler de volta com read_parquet — sem carregar em tabela.
import os
caminho = "/tmp/vendas.parquet"
vendas.to_parquet(caminho, index=False)
print("Parquet gravado:", caminho, "(", os.path.getsize(caminho), "bytes )")

print("\nDuckDB lendo o Parquet DIRETO do disco e agregando:")
duckdb.sql(f"""
    SELECT categoria_id, SUM(valor) AS faturamento
    FROM read_parquet('{caminho}')
    GROUP BY categoria_id
    ORDER BY faturamento DESC
""").df()

Parquet gravado: /tmp/vendas.parquet ( 3644 bytes )

DuckDB lendo o Parquet DIRETO do disco e agregando:


,categoria_id,faturamento
0,10,2050.0
1,40,440.0
2,20,201.6
3,30,45.0


### Quando empurrar o trabalho para o banco

- **Empurre para o SQL:** filtros, agregações, JOINs, ordenações sobre volumes grandes — o banco move menos dados e processa em colunar.
- **Mantenha em Python:** orquestração, lógica condicional complexa, chamadas a APIs, integração com bibliotecas.

> A regra do encontro: **filtre e agregue o mais cedo e o mais perto da fonte possível.** Trazer um milhão de linhas para a memória só para somá-las é desperdício — peça a soma ao banco e traga uma linha.

A tendência de mercado reforça isso: **DuckDB** (o "SQLite do OLAP", analytics in-process sobre Parquet) e **dbt** (transformação como SQL versionado, modular e testado). Veja a caixa de tendência no `.md`.

---

## 🧪 Exercícios do Encontro 3

Resolva no próprio notebook, criando novas células abaixo. Use sempre `con.sql(""" ... """).df()`.

1. **SELECT + WHERE.** Liste as vendas da cidade *Fortaleza* (dica: precisa de um `JOIN` com `clientes`), ordenadas por valor decrescente. Quantas são?
2. **Agregação + HAVING.** Some o faturamento por **segmento** de cliente (`varejo` vs. `corporativo`) e mantenha só os segmentos com mais de 3 vendas.
3. **JOIN + CASE WHEN.** Junte a fato com `categorias` e classifique cada venda por **departamento**; conte quantas vendas há por departamento.
4. **Window + CTE.** Reescreva o ranking da seção 5 trocando `RANK()` por `DENSE_RANK()` e, numa CTE seguinte, filtre só o **top 3** (`WHERE posicao <= 3`). Por que o filtro precisa estar fora da janela?

> Dica: o melhor exercício é **quebrar a consulta de propósito** (esqueça uma coluna no `GROUP BY`, troque um `INNER` por `LEFT`) e observar o que muda. Erros são o melhor professor de SQL.

---

## Encerramento

Este notebook acompanha o documento `.md` (teoria detalhada) e os slides `.pptx` (aula expositiva) do Encontro 3.

Hoje você: montou um **modelo estrela** (fato + dimensões) em DuckDB, trocou a mentalidade procedural pela **declarativa**, dominou `SELECT/WHERE/ORDER BY/LIMIT`, **agregou** com `GROUP BY/HAVING/CASE WHEN`, costurou tabelas com os **quatro JOINs**, ranqueou clientes com **window functions + CTEs**, e consultou **DataFrames e Parquet** com SQL via DuckDB.

No **Encontro 4** saímos da "máquina só" e entramos em **processamento em escala**: o que muda quando o dado não cabe na memória.

**Bibliografia:** McKinney, *Python para Análise de Dados* (2022) · Reis & Housley, *Fundamentos de Engenharia de Dados* (2023) · Kleppmann, *Designing Data-Intensive Applications* (2017).